# Exercise 2: Save Deduplicated Results to Apache Iceberg

## Learning Objectives

In this exercise, you will:
- Connect Spark to an **Apache Iceberg REST catalog** (Cloudera HMS REST Catalog)
- Load the deduplicated Parquet output from `01_Basic_Deduplication.ipynb`
- Write those results into an Iceberg table
- Find and verify the table through the REST catalog

## Prerequisites

1. Run **`00_Getting_Started.ipynb`** and **`01_Basic_Deduplication.ipynb`** first.
2. Notebook 01 writes:

```text
hdfs:///tmp/cdp_user_demo/phase1/results/exercise1_exact.parquet
```

3. Have Iceberg REST catalog connection details available:
- REST catalog URI
- OAuth credential (`clientId:secret`) when required by Cloudera Data Sharing
- Target namespace / warehouse settings for your environment


## Step 1: Configure Iceberg REST Catalog

Set catalog connection values below (or via environment variables).  
Cloudera REST catalog URIs typically look like:

```text
https://<datalake-hostname>/<datalake-name>/cdp-datashare-access/hms-api
```


In [ ]:
import os

# --- Edit these for your Cloudera environment (env vars override defaults) ---
CATALOG_NAME = os.environ.get("ICEBERG_CATALOG_NAME", "iceberg")
REST_URI = os.environ.get(
    "ICEBERG_REST_URI",
    "https://<DATALAKE-HOSTNAME>/<DATALAKE-NAME>/cdp-datashare-access/hms-api",
)
# Format: clientId:secret (from Cloudera Data Share consumer credentials)
REST_CREDENTIAL = os.environ.get("ICEBERG_REST_CREDENTIAL", "")

NAMESPACE = os.environ.get("ICEBERG_NAMESPACE", "cdp_user_demo")
TABLE_NAME = os.environ.get("ICEBERG_TABLE", "deduped_customers")
FULL_TABLE = f"{CATALOG_NAME}.{NAMESPACE}.{TABLE_NAME}"

# Input from Exercise 1
HDFS_INPUT = "hdfs:///tmp/cdp_user_demo/phase1/results/exercise1_exact.parquet"

print(f"Catalog:     {CATALOG_NAME}")
print(f"REST URI:    {REST_URI}")
print(f"Namespace:   {NAMESPACE}")
print(f"Table:       {TABLE_NAME}")
print(f"Full name:   {FULL_TABLE}")
print(f"HDFS input:  {HDFS_INPUT}")
print(f"Credential:  {'set' if REST_CREDENTIAL else 'NOT SET — update REST_CREDENTIAL / ICEBERG_REST_CREDENTIAL'}")


## Step 2: Create Spark Session with Iceberg REST Catalog

In Cloudera AI Workbench, Iceberg runtime jars are often already on the cluster classpath.  
If `org.apache.iceberg` classes are missing, add the Iceberg Spark runtime package appropriate for your Spark version.


In [ ]:
from pyspark.sql import SparkSession

builder = (
    SparkSession.builder
    .appName("Exercise2_IcebergRESTCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .config("spark.sql.defaultCatalog", CATALOG_NAME)
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "rest")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", REST_URI)
)

# OAuth credential for Cloudera Iceberg REST Catalog / Data Sharing
if REST_CREDENTIAL:
    builder = builder.config(f"spark.sql.catalog.{CATALOG_NAME}.credential", REST_CREDENTIAL)

# Optional: set default namespace when supported by your catalog
builder = builder.config(
    f"spark.sql.catalog.{CATALOG_NAME}.default-namespace", NAMESPACE
)

spark = builder.getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark master:  {spark.sparkContext.master}")
print(f"Default catalog config: spark.sql.defaultCatalog={CATALOG_NAME}")
print("✓ Spark session created with Iceberg REST catalog")


## Step 3: Load Deduplicated Results from Exercise 1


In [ ]:
df = spark.read.parquet(HDFS_INPUT)

print(f"✓ Loaded: {HDFS_INPUT}")
print(f"Rows: {df.count():,}")
print(f"Columns: {', '.join(df.columns)}")
df.show(10, truncate=False)
df.printSchema()


## Step 4: Create Namespace and Write Iceberg Table

Create the namespace (database) if needed, then write the DataFrame as an Iceberg table through the REST catalog.


In [ ]:
# Create namespace in the REST catalog if it does not exist
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {CATALOG_NAME}.{NAMESPACE}")
print(f"✓ Namespace ready: {CATALOG_NAME}.{NAMESPACE}")

# Write / replace Iceberg table from Exercise 1 results
(
    df.writeTo(FULL_TABLE)
    .using("iceberg")
    .tableProperty("write.format.default", "parquet")
    .createOrReplace()
)

print(f"✓ Iceberg table written: {FULL_TABLE}")


## Step 5: Find the Table in the Iceberg REST Catalog

Use Spark SQL (backed by the REST catalog) to list namespaces/tables and confirm the new table is discoverable.


In [ ]:
print("=== Namespaces in REST catalog ===")
spark.sql(f"SHOW NAMESPACES IN {CATALOG_NAME}").show(truncate=False)

print("=== Tables in namespace ===")
tables_df = spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.{NAMESPACE}")
tables_df.show(truncate=False)

# Confirm our table is present
table_names = [r.tableName if hasattr(r, "tableName") else r["tableName"] for r in tables_df.collect()]
# Spark may return column as 'tableName' or 'name' depending on version
if not table_names:
    table_names = [r.asDict().get("tableName") or r.asDict().get("name") for r in tables_df.collect()]

found = TABLE_NAME in table_names or any(TABLE_NAME == str(n) for n in table_names)
print(f"Looking for table: {TABLE_NAME}")
print(f"Tables found: {table_names}")
print("✓ Table found in Iceberg REST catalog" if found else "✗ Table NOT found — check write step / namespace")


In [ ]:
print("=== DESCRIBE TABLE ===")
spark.sql(f"DESCRIBE TABLE EXTENDED {FULL_TABLE}").show(100, truncate=False)

print("=== Sample query from Iceberg table ===")
iceberg_df = spark.table(FULL_TABLE)
print(f"Rows in Iceberg table: {iceberg_df.count():,}")
iceberg_df.show(10, truncate=False)


## Step 6 (Optional): Query the REST Catalog HTTP API Directly

This calls the Iceberg REST Catalog OpenAPI endpoints used by engines to discover tables.  
Skip if your environment blocks outbound HTTP from the session or requires additional Knox headers.


In [ ]:
import json
import urllib.request
import urllib.error
import base64

def rest_get(path: str):
    """GET a path under the Iceberg REST catalog URI."""
    url = REST_URI.rstrip("/") + path
    req = urllib.request.Request(url, method="GET")
    req.add_header("Accept", "application/json")

    # Many Cloudera deployments expect OAuth; basic clientId:secret may work for token exchange.
    # If your REST endpoint already accepts Bearer tokens, set ICEBERG_REST_TOKEN instead.
    token = os.environ.get("ICEBERG_REST_TOKEN", "")
    if token:
        req.add_header("Authorization", f"Bearer {token}")
    elif REST_CREDENTIAL:
        encoded = base64.b64encode(REST_CREDENTIAL.encode("utf-8")).decode("ascii")
        req.add_header("Authorization", f"Basic {encoded}")

    with urllib.request.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read().decode("utf-8"))

try:
    # List tables in the namespace via REST Catalog API
    # Spec: GET /v1/namespaces/{namespace}/tables
    ns_path = NAMESPACE.replace(".", "%1F")  # multipart namespace separator per Iceberg REST spec
    payload = rest_get(f"/v1/namespaces/{ns_path}/tables")
    identifiers = payload.get("identifiers", [])
    print("REST API tables in namespace:")
    print(json.dumps(identifiers, indent=2))

    matched = [
        i for i in identifiers
        if i.get("name") == TABLE_NAME or TABLE_NAME in str(i)
    ]
    if matched:
        print(f"\n✓ Found via REST API: {matched}")
        # Load table metadata
        meta = rest_get(f"/v1/namespaces/{ns_path}/tables/{TABLE_NAME}")
        print("\nTable metadata keys:", list(meta.keys()))
    else:
        print(f"\n✗ Table '{TABLE_NAME}' not returned by REST list endpoint")
except urllib.error.HTTPError as e:
    print(f"REST API HTTP error: {e.code} {e.reason}")
    print("Spark SQL discovery in Step 5 may still succeed; adjust auth headers/URI for direct REST calls.")
except Exception as e:
    print(f"REST API call skipped/failed: {e}")
    print("Use Step 5 (SHOW TABLES) as the primary catalog verification in Cloudera AI Workbench.")


## Summary

| Item | Value |
|------|-------|
| Source (Exercise 1) | `hdfs:///tmp/cdp_user_demo/phase1/results/exercise1_exact.parquet` |
| Iceberg table | `iceberg.cdp_user_demo.deduped_customers` (defaults) |
| Catalog type | Iceberg REST (`spark.sql.catalog.*.type=rest`) |

### Key Takeaways

- Deduplicated lab results can be promoted from HDFS staging files into governed Iceberg tables
- The Iceberg REST catalog makes tables discoverable to Spark and other REST-capable engines
- `SHOW TABLES` / `DESCRIBE TABLE` confirm registration; the REST `/v1/namespaces/.../tables` API is the engine-facing discovery path

## Cleanup


In [ ]:
spark.stop()
print("✓ Spark session stopped")
